References
https://tiktokenizer.vercel.app/?model=gpt2


In [12]:
#!pip install tiktoken

In [22]:
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


In [23]:
tokanizer=tiktoken.get_encoding('gpt2')

In [21]:
with open('the-verdict.txt',"r",encoding="utf-8") as f:
         booktext=f.read()
         print(tokanizer.encode(booktext,allowed_special={'<|endoftext|>'}))




[40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026, 15632, 438, 2016, 257, 922, 5891, 1576, 438, 568, 340, 373, 645, 1049, 5975, 284, 502, 284, 3285, 326, 11, 287, 262, 6001, 286, 465, 13476, 11, 339, 550, 5710, 465, 12036, 11, 6405, 257, 5527, 27075, 11, 290, 4920, 2241, 287, 257, 4489, 64, 319, 262, 34686, 41976, 13, 357, 10915, 314, 2138, 1807, 340, 561, 423, 587, 10598, 393, 28537, 2014, 198, 198, 1, 464, 6001, 286, 465, 13476, 1, 438, 5562, 373, 644, 262, 1466, 1444, 340, 13, 314, 460, 3285, 9074, 13, 46606, 536, 5469, 438, 14363, 938, 4842, 1650, 353, 438, 2934, 489, 3255, 465, 48422, 540, 450, 67, 3299, 13, 366, 5189, 1781, 340, 338, 1016, 284, 3758, 262, 1988, 286, 616, 4286, 705, 1014, 510, 26, 475, 314, 836, 470, 892, 286, 326, 11, 1770, 13, 8759, 2763, 438, 1169, 2994, 284, 943, 17034, 318, 477, 314, 892, 286, 526, 383, 1573, 11, 319, 9074, 13, 536, 5469, 338, 11914, 11, 33096, 663, 4808, 3808, 62, 355, 996, 484, 547, 12548, 287, 281, 13079, 410, 12523, 286, 

In [47]:
class datasetgen(Dataset):
    def __init__(self,txt:str,context_length:int,stride:int):
        super().__init__()
        self.input_ids=[]
        self.target_ids=[]
        self.text=txt
        self.context_length=context_length
        self.stride=stride
    def prepare(self):
        txt_encoded=tokanizer.encode(self.text)
        for i in range(0,len(txt_encoded)-self.context_length,self.stride):
            in_ids=[]
            targ_ids=[]
            for j in range(i,i+self.context_length):
                #print(f"{j}:{i}")
                in_ids.append(txt_encoded[j])
                targ_ids.append(txt_encoded[j+1])
            self.input_ids.append(in_ids)
            self.target_ids.append(targ_ids)

In [49]:
datagen=datasetgen(booktext,10,1)
datagen.prepare()

In [50]:
print(datagen.input_ids)
print(datagen.target_ids)


[[40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138], [367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257], [2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026], [1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026, 15632], [1807, 3619, 402, 271, 10899, 2138, 257, 7026, 15632, 438], [3619, 402, 271, 10899, 2138, 257, 7026, 15632, 438, 2016], [402, 271, 10899, 2138, 257, 7026, 15632, 438, 2016, 257], [271, 10899, 2138, 257, 7026, 15632, 438, 2016, 257, 922], [10899, 2138, 257, 7026, 15632, 438, 2016, 257, 922, 5891], [2138, 257, 7026, 15632, 438, 2016, 257, 922, 5891, 1576], [257, 7026, 15632, 438, 2016, 257, 922, 5891, 1576, 438], [7026, 15632, 438, 2016, 257, 922, 5891, 1576, 438, 568], [15632, 438, 2016, 257, 922, 5891, 1576, 438, 568, 340], [438, 2016, 257, 922, 5891, 1576, 438, 568, 340, 373], [2016, 257, 922, 5891, 1576, 438, 568, 340, 373, 645], [257, 922, 5891, 1576, 438, 568, 340, 373, 645, 1049], [922, 5891, 1576, 438, 568, 340, 373, 645, 1049, 5975], [5891, 157